# PPEM Application Profiling Vizualizer 

Author: K. Desnos & MAI-Code-1.1-Flash  
Version : 1.0 (Sep. 2026)

## 0. Profile the application
1. Make sure the `ENABLE_PROFILING` is enabled for your C project.   
   In VSCode, press `[F1]`, search for `CMake: Edit CMake Cache (UI)`, and set the `ENABLE_PROFILING` to true.
2. Clean, Build and Run the executable in debug or release config.
3. Run the executable, a `trace.out` file should be generated in the execution directory. 

## 1. Configuration & trace parsing
In the following cell, indicate the location of the trace file and the associated executable.
The given files will be parsed to extract profiling info.

In [2]:
trace_file = "../build/Release/trace.out"  # Replace with the actual path to your trace file
executable = "../build/Release/mandelbrot.exe"  # Replace with the actual path to your executable

# Parse the given files
import trace_parser
calls, cores = trace_parser.parse_trace(trace_file, executable)

# Print number of function calls
print(f"Total function calls: {len(calls)}")

Total function calls: 1063


## 2. Analyze profiling data

### 2.1. Call Tree
This first array displays the call tree extracted from the graph. Each function call appears in the hierarchy of function call, with its given statistics.

- `call_count`: number of times this function was executed.
- `total_cpu_time_including_children`: cumulative execution time for all calls of this function, including the time spent in callees.
- `total_cpu_time_excluding_children`: cumulative execution time spent directly in this function body, excluding callees.
- `avg_cpu_time_including_children`: average duration per call, including callee time. <!-- `total_cpu_time_including_children` / `call_count`.!-->
- `avg_cpu_time_excluding_children`: average duration per call, excluding callee time. <!-- `total_cpu_time_excluding_children` / `call_count`. -->
- `percent_of_parent_incl`: how much of the parent function’s inclusive time is due to this function. <!-- `total_cpu_time_including_children` / `parent_total_cpu_time_including_children` * 100. -->
- `percent_of_parent_excl`: how much of the parent function’s exclusive time is due to this function. <!-- `total_cpu_time_excluding_children` / `parent_total_cpu_time_including_children` * 100. -->
- `percent_of_total_incl`: percentage of the total CPU time attributed to this function, including the time spent in callees.
- `percent_of_total_excl`: percentage of the total CPU time attributed to this function, excluding the time spent in callees.


These values help identify which functions or call branches dominate total runtime and which parts are mainly overhead versus actual work.

In [3]:
# Build a call tree from the parsed calls
call_tree_roots = trace_parser.analyze_call_tree(calls)
call_tree_df = trace_parser.build_call_tree_dataframe(call_tree_roots)

# Choose the columns you want to display, and set the order here.
# Edit this list to keep only the metrics you care about.
selected_columns = [
    "call_count",
    "percent_of_total_incl",
    "percent_of_total_excl",
    "total_cpu_time_including_children",
    "total_cpu_time_excluding_children",
    "avg_cpu_time_including_children",
    "avg_cpu_time_excluding_children",
    "percent_of_parent_incl",
    "percent_of_parent_excl"
]

# Keep only the existing columns, in the desired order.
visible_columns = [col for col in selected_columns if col in call_tree_df.columns]
call_tree_df = call_tree_df[visible_columns]

call_tree_df

,call_count,percent_of_total_incl,percent_of_total_excl,total_cpu_time_including_children,total_cpu_time_excluding_children,avg_cpu_time_including_children,avg_cpu_time_excluding_children,percent_of_parent_incl,percent_of_parent_excl
display_name,,,,,,,,,
main,1,100.00,0.21,13618386,28155,13618386.00,28155.00,100.00,0.21
├── yuvDisplayInit,1,0.66,0.66,90348,90343,90348.00,90343.00,0.66,0.66
│ └── startTiming,4,0.00,0.00,5,5,1.25,1.25,0.01,0.01
├── startTiming,2,0.00,0.00,3,3,1.50,1.50,0.00,0.00
├── fillMandelbrot,95,91.76,91.76,12495805,12495805,131534.79,131534.79,91.76,91.76
├── MD5_Update,95,3.66,0.02,498696,3254,5249.43,34.25,3.66,0.02
│ ├── MD5_Init,95,0.00,0.00,49,49,0.52,0.52,0.01,0.01
│ ├── body,95,3.64,3.64,495258,495258,5213.24,5213.24,99.31,99.31
│ └── MD5_Final,95,0.00,0.00,135,77,1.42,0.81,0.03,0.02


### 2.1 Flame Graph view of the function call tree

This flame graph represents the inclusive CPU time of each function in the call tree. The wider a block, the more time it spent including its descendants.

In [4]:
# Plot the call hierarchy as an inclusive-time flame graph
flame_graph = trace_parser.build_flamegraph(calls)
flame_graph

## 3 Per function statistics

### 3.1. Function statistics
This array gathers all statistics for each encountered function call such that a function calls encountered in several places in the profiles program are summed.

In [5]:
# Aggregate all occurrences of the same function across the whole profile.
# This sums the values of repeated calls encountered at different places in the program.
function_stats_df = trace_parser.build_function_statistics_dataframe(calls)

# Choose the columns you want to display, and set the order here.
# Edit this list to keep only the metrics you care about.
selected_columns = [
    "call_count",
    "percent_of_total_incl",
    "percent_of_total_excl",
    "total_cpu_time_including_children",
    "total_cpu_time_excluding_children",
    "avg_cpu_time_including_children",
    "avg_cpu_time_excluding_children",
    "function_file",
    "function_address"
]

# Keep only the existing columns, in the desired order.
visible_columns = [col for col in selected_columns if col in function_stats_df.columns]
function_stats_df = function_stats_df[visible_columns]

function_stats_df

,call_count,percent_of_total_incl,percent_of_total_excl,total_cpu_time_including_children,total_cpu_time_excluding_children,avg_cpu_time_including_children,avg_cpu_time_excluding_children,function_file,function_address
function_name,,,,,,,,,
main,1,100.00,0.21,13618386,28155,13618386.00,28155.00,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,00000001400048A0
yuvDisplayInit,1,0.66,0.66,90348,90343,90348.00,90343.00,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,0000000140003310
startTiming,101,0.00,0.00,244,244,2.42,2.42,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,00000001400014E0
fillMandelbrot,95,91.76,91.76,12495805,12495805,131534.79,131534.79,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,00000001400015C0
MD5_Update,95,3.66,0.02,498696,3254,5249.43,34.25,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,0000000140002AA0
MD5_Init,95,0.00,0.00,49,49,0.52,0.52,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,0000000140001FC0
body,190,3.64,3.64,495316,495316,2606.93,2606.93,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,0000000140002010
MD5_Final,95,0.00,0.00,135,77,1.42,0.81,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,00000001400027D0
mandelbrotToYUV,95,2.67,2.67,364078,364078,3832.40,3832.40,C:/Users/HP/Documents/PPEM/proj/projet_ppem-ma...,0000000140001920


### 3.2 Function execution time distribution

The following chart allows to visualize the distribution of execution time for each function. 

In [6]:
# Select a subset of functions to display in the box plot.
# Use the function names exactly as they appear in the trace/profile table.
# Set this to None to display all functions.
selected_boxplot_functions = [
    "thread_wrapper",
    "yuvDisplay",
    "body"
]

# Keep only the selected functions; if a name is missing from the trace, it is ignored.
box_plot = trace_parser.build_boxplot(calls, selected_boxplot_functions)
box_plot